### 1.Generate three tables for a fictional college fest in a notebook named 'fest_merge.ipynb' with `np.random.seed(42)`: students_df (student_id 1 to 60, college from 5 invented names, year 1 to 4), events_df (event_id 1 to 8 with event_name, category Tech, Cultural or Sports, and venue) and reg_df (150 unique student_id and event_id pairs with student_id from 1 to 65, plus reg_id and ticket_inr 50 to 500). Save 'fest_students.csv', 'fest_events.csv' and 'fest_registrations.csv'. Hint: draw 200 pairs, `drop_duplicates(subset=['student_id', 'event_id'])`, then `head(150)`

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

students_df = pd.DataFrame({
    'student_id': np.arange(1, 61),
    'college': np.random.choice(
        ['ABC College', 'XYZ College', 'GIT College',
         'Modern College', 'National College'], 60
    ),
    'year': np.random.randint(1, 5, 60)
})

events_df = pd.DataFrame({
    'event_id': np.arange(1, 9),
    'event_name': [
        'Coding Competition', 'Dance Battle',
        'Cricket Match', 'Quiz Competition',
        'Singing Contest', 'Hackathon',
        'Football Match', 'Drama Competition'
    ],
    'category': [
        'Tech', 'Cultural', 'Sports', 'Tech',
        'Cultural', 'Tech', 'Sports', 'Cultural'
    ],
    'venue': [
        'Hall A', 'Auditorium', 'Ground 1', 'Hall B',
        'Auditorium', 'Lab 1', 'Ground 2', 'Hall C'
    ]
})

reg_df = pd.DataFrame({
    'student_id': np.random.randint(1, 66, 200),
    'event_id': np.random.randint(1, 9, 200)
})

reg_df = reg_df.drop_duplicates(
    subset=['student_id', 'event_id']
).head(150).copy()

reg_df.insert(0, 'reg_id', np.arange(1, len(reg_df) + 1))
reg_df['ticket_inr'] = np.random.randint(50, 501, len(reg_df))

students_df.to_csv('fest_students.csv', index=False)
events_df.to_csv('fest_events.csv', index=False)
reg_df.to_csv('fest_registrations.csv', index=False)

print("STUDENTS DATA")
print(students_df.head())

print("EVENTS DATA")
print(events_df.head())

print("REGISTRATIONS DATA")
print(reg_df.head())

# Check shapes
print("Dataset Shapes:")
print("Students:", students_df.shape)
print("Events:", events_df.shape)
print("Registrations:", reg_df.shape)

print("All CSV files saved successfully!")

STUDENTS DATA
   student_id           college  year
0           1    Modern College     2
1           2  National College     2
2           3       GIT College     2
3           4  National College     2
4           5  National College     4
EVENTS DATA
   event_id          event_name  category       venue
0         1  Coding Competition      Tech      Hall A
1         2        Dance Battle  Cultural  Auditorium
2         3       Cricket Match    Sports    Ground 1
3         4    Quiz Competition      Tech      Hall B
4         5     Singing Contest  Cultural  Auditorium
REGISTRATIONS DATA
   reg_id  student_id  event_id  ticket_inr
0       1           5         2         386
1       2          41         7         139
2       3          28         1         313
3       4           7         4         142
4       5          12         5         459
Dataset Shapes:
Students: (60, 3)
Events: (8, 4)
Registrations: (150, 4)
All CSV files saved successfully!


### 2.Build 'join_counts', a 4-row DataFrame with the columns how and row_count, by merging reg_df with students_df on student_id using `how` set to inner, left, right and outer. Print it. Hint: the left count equals `len(reg_df)` and the outer count is the largest.

In [2]:

join_counts = pd.DataFrame([                                                                                      # Calculate row counts for different join types
    {'how': 'inner', 'row_count': len(pd.merge(reg_df, students_df, on='student_id', how='inner'))},
    {'how': 'left', 'row_count': len(pd.merge(reg_df, students_df, on='student_id', how='left'))},
    {'how': 'right', 'row_count': len(pd.merge(reg_df, students_df, on='student_id', how='right'))},
    {'how': 'outer', 'row_count': len(pd.merge(reg_df, students_df, on='student_id', how='outer'))}
])

print(join_counts)

     how  row_count
0  inner        140
1   left        150
2  right        142
3  outer        152


### 3.Create 'attendance_df' from `reg_df.sample(100, random_state=42)` keeping student_id and event_id plus a new 'attended' column of 0 or 1, then merge reg_df with it on both keys (`on=['student_id', 'event_id']`, `how='left'`) into 'reg_att'. Check that its row count equals `len(reg_df)` and that attended is NaN for exactly 50 rows.

In [3]:
attendance_df = reg_df.sample(100, random_state=42)[['student_id', 'event_id']].copy()

attendance_df['attended'] = np.random.choice([0, 1], 100)

reg_att = pd.merge(
    reg_df,
    attendance_df,
    on=['student_id', 'event_id'],
    how='left'
)

print("reg_df rows:", len(reg_df))
print("reg_att rows:", len(reg_att))

print("Missing attendance:", reg_att['attended'].isna().sum())

print(reg_att.head())

reg_df rows: 150
reg_att rows: 150
Missing attendance: 50
   reg_id  student_id  event_id  ticket_inr  attended
0       1           5         2         386       0.0
1       2          41         7         139       NaN
2       3          28         1         313       NaN
3       4           7         4         142       NaN
4       5          12         5         459       0.0


### 4 .Create 'events_full' by joining events_df with a new 8-row DataFrame 'event_budget_df' (columns budget_inr and volunteers) using `DataFrame.join`, then left-merge 'reg_att' with students_df and events_full into 'fest_master' and save it as 'fest_master.csv' with `index=False`. Its row count must be 150. Hint: `join` matches on the index, so both frames need the same row order.

In [4]:
event_budget_df = pd.DataFrame({
    'budget_inr': np.random.randint(5000, 20001, 8),
    'volunteers': np.random.randint(5, 21, 8)
})

events_full = events_df.join(event_budget_df)

fest_master = pd.merge(reg_att,students_df,on='student_id',how='left')

fest_master = pd.merge(fest_master,events_full,on='event_id',how='left')

fest_master.to_csv('fest_master.csv', index=False)

print(fest_master.head())
print("Total rows:", len(fest_master))
print("CSV file saved successfully!")

   reg_id  student_id  event_id  ticket_inr  attended           college  year  \
0       1           5         2         386       0.0  National College   4.0   
1       2          41         7         139       NaN    Modern College   1.0   
2       3          28         1         313       NaN       XYZ College   4.0   
3       4           7         4         142       NaN       GIT College   1.0   
4       5          12         5         459       0.0       GIT College   2.0   

           event_name  category       venue  budget_inr  volunteers  
0        Dance Battle  Cultural  Auditorium       12026          18  
1      Football Match    Sports    Ground 2       19785          11  
2  Coding Competition      Tech      Hall A       10713          12  
3    Quiz Competition      Tech      Hall B        9804          15  
4     Singing Contest  Cultural  Auditorium        6838          17  
Total rows: 150
CSV file saved successfully!
